# Suite FRM — The framework

Every dialect implements the same protocols (`@mbse/expressions/Framework`): its expressions are serializable
through meta-schemas and builders, structurally traversable through forms, validatable, evaluatable and
translatable. `Framework.Terms.Declared` derives most of a dialect from its kinds' data classes; this suite
checks that machinery on every dialect at once. The dialects' own rules are suite DIA, and translation suite TRN.

In [ ]:
import { Evaluators as V, Expressions as E } from "@mbse/expressions";
import * as Ccpp from "@mbse/expressions/Dialects/Ccpp";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python from "@mbse/expressions/Dialects/Python";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import { Domains as D, Errors as FE, Evaluators as FV, Symbolics as SY, Terms as F, Translators as FT } from "@mbse/expressions/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { Errors, JSON, Proxies, YAML } from "@mbse/schemas/Framework";
import { register } from "@mbse/expressions";

const store = register(new Proxies.OfStore()); // holds every dialect's meta-schemas

const { KeyError, NotImplementedError, ValueError } = Errors;
import { assert, conformance_problems, raises, same, set_property } from "./support.js";

const [P, M, X, L, C] = [Python.Expressions, Matlab.Expressions, Excel.Expressions, Latex.Expressions, Ccpp.Expressions];
const [PV, MV, XV] = [Python.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const SV = SystemVerilog.Expressions;
const DIALECTS = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT, C.DIALECT, SV.DIALECT];
const EVALUATORS = new Map<F.Declared, unknown>([[E.DIALECT, V.OfAny], [P.DIALECT, PV.OfAny], [M.DIALECT, MV.OfAny],
  [X.DIALECT, XV.OfAny], [C.DIALECT, Ccpp.Evaluators.OfAny], [SV.DIALECT, SystemVerilog.Evaluators.OfAny]]); // Latex has none
const text = (value: unknown) => globalThis.JSON.stringify(value, (_k, v) => (typeof v === "bigint" ? `${v}n` : v));
const self = E.variable("this");
const basic = E.let_("a", self.age, E.variable("a").ge(18n).and_(self.has("email"))).data;
const samples: Record<string, any> = { // one expression per dialect, with every kind
  Basic: E.operation("and", basic, self.items.all("x", E.variable("x").gt(0n))).data, // a quantifier too
  Python: P.import_("numpy", P.importfrom("math", "floor", P.let_("a", P.name("this"), P.ifexp( // every kind
    P.boolop("or", P.boolop("and", P.unaryop("not", P.compare("==", P.attribute(P.name("a"), "z"), 1n)),
      P.unaryop("-", P.attribute(P.name("a"), "w"))), P.binop("*", 2n, 3n)),
    P.call("np.add", P.attribute(P.name("a"), "x"), P.subscript(P.name("a"), "y")), P.call("f", P.generator("p", P.attribute(P.name("a"), "z"), P.index(P.name("p"), 0n), P.name("p"))))),
  "f"), "np"),
  Matlab: M.import_("pkg.*", M.binary("&&", M.unary("~", M.field(M.identifier("this"), "age")),
    M.call("isfield", "x", M.index(M.arrayfun("p", M.identifier("this"), M.identifier("p")), 1n)))),
  Excel: X.let_("a", X.field(X.name("this"), "age"), X.function("AND", X.infix(">=", X.name("a"), X.cell("A1")),
    X.prefix("-", X.function("SUM", X.map_("p", X.name("a"), X.infix("*", X.name("p"), 2n)))))),
  Latex: L.where("a", L.member(L.symbol("this"), "age"), L.binary("\\land", L.binary("\\geq", L.symbol("a"), 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", L.symbol("this"), "email")), L.binary(">", L.frac(L.symbol("a"), 2n), 1.5)))),
  Ccpp: C.conditional(C.binary("&&", C.unary("!", C.member(C.identifier("this"), "a")), C.binary( // every kind
    ">", C.subscript(C.member(C.identifier("this"), "b", "->"), 0n), C.constant(1.5, "float"))),
    C.cast("uint8_t", C.call("f", 1n)), C.constant(5n, "unsigned int")),
  SystemVerilog: SV.binary("->", SV.inside(SV.member(SV.identifier("this"), "a"), 1n, SV.span(2n, 5n)), SV.conditional( // every kind
    SV.unary("!", SV.select(SV.identifier("this"), 0n)), SV.concatenation(SV.replication(2n, SV.vector("10")), SV.range_(SV.identifier("this"), 3n, 0n)),
    SV.cast("byte", SV.call("$clog2", SV.iterate(SV.method(SV.identifier("this"), "unique"), "sum", "p", SV.identifier("p")))))),
};
const kindOf = (dialect: F.Declared, tag: string) => dialect.kinds().get(tag) as any;
const bytes = (...values: number[]) => new Uint8Array(values);

/** Python's `==` on forms: the same kind, attributes and arguments. */
function sameForm(a: F.Form, b: F.Form): boolean {
  return a.kind === b.kind && a.attributes.size === b.attributes.size
    && [...a.attributes].every(([k, v]) => b.attributes.has(k) && F.sameNative(v, b.attributes.get(k)))
    && same(a.arguments, b.arguments);
}

## FRM-01 · Every dialect, expression, evaluator and translator implements its protocol

In [ ]:
{
  assert([...EVALUATORS.values()].every((evaluate) => typeof evaluate === "function"));
  for (const dialect of DIALECTS) {
    assert(conformance_problems(F.Declared, "Dialect").length === 0);
    for (const kind of dialect.kinds().values()) {
      assert(conformance_problems(kind, "Expression").length === 0, kind.name);
      assert(conformance_problems(kind, "Visitable").length === 0, kind.name);
      assert(conformance_problems(dialect.builders.get(kind.KIND) as any, "OfObject").length === 0, kind.name);
    }
    for (const expression of F.walk(samples[dialect.name()])) assert(expression.dialect() === dialect);
  }
  assert(same(DIALECTS.map((d) => d.name()), ["Basic", "Python", "Matlab", "Excel", "Latex", "Ccpp", "SystemVerilog"]) && String(E.DIALECT) === "<dialect Basic>");
  assert(same([...X.DIALECT.kinds().keys()], ["constant", "name", "cell", "let", "function", "infix", "prefix", "field", "map"]));
  for (const translator of Translators.TRANSLATORS) {
    assert(translator instanceof FT.Pairwise && conformance_problems(FT.Pairwise, "Translator").length === 0);
  }
  for (const domain of [new D.OfTypes("int", BigInt), new D.OfValues("odd", (v) => Number(v) % 2 === 1), new D.OfUnion(D.Anything)]) {
    assert(conformance_problems(domain.constructor as any, "Domain").length === 0);
  }
  for (const signature of [new D.Function([], D.Anything), new D.Same(1, [D.Anything]), new D.Overloaded(new D.Function([], D.Anything))]) {
    assert(conformance_problems(signature.constructor as any, "Signature").length === 0);
  }
}

## FRM-02 · Forms and `make` are inverses, for every kind of every dialect

In [ ]:
{
  for (const dialect of DIALECTS) {
    const kinds = new Set<string>();
    for (const expression of F.walk(samples[dialect.name()])) {
      const form = expression.form();
      kinds.add(form.kind);
      const made = dialect.make(form);
      assert(made.constructor === expression.constructor && sameForm(made.form(), form) && made !== expression);
    }
    assert(same([...kinds].sort(), [...dialect.kinds().keys()].sort()), dialect.name()); // the samples have every kind
  }
  assert(sameForm(E.OfLiteral.resolve(3n).form(), new F.Form("literal", new Map([["value", 3n]]))));
  assert(sameForm(new E.OfLet.Data("x").form(), new F.Form("let", new Map([["name", "x"]]), [null, null]))); // empty slots are null
  assert(sameForm(new E.OfOperation.Data().form(), new F.Form("operation")));
  raises(ValueError, () => E.DIALECT.make(new F.Form("constant")), "Basic has no kind 'constant'");
  raises(ValueError, () => E.DIALECT.make(new F.Form("literal", new Map([["name", "x"]]))), "a literal has no attribute 'name'");
  raises(ValueError, () => E.DIALECT.make(new F.Form("let", new Map([["name", "x"]]), [basic])), "a let takes 2 arguments, got 1");
  raises(ValueError, () => P.DIALECT.make(new F.Form("call")), "a call takes at least 1 arguments, got 0"); // its function, then its arguments
  const made = P.DIALECT.make(new F.Form("call", new Map(), [P.name("f"), P.constant(1n), P.constant(2n)]));
  assert(made.function.name === "f" && same(made.arguments.map((a: any) => a.value), [1n, 2n]));
  assert(E.DIALECT.literal([1n] as any) === null && X.DIALECT.literal(bytes(120)) === null); // no literal kind holds these
  assert(M.DIALECT.resolve(1n).value === 1n && M.DIALECT.resolve(true).kind().KIND === "constant");
  raises(TypeError, () => X.DIALECT.resolve(bytes(120)), "expected an expression, a native value or a callable taking its builder, got b'x'");
}

## FRM-03 · `walk`, `fold` and `same` traverse through forms alone

In [ ]:
{
  const shared = E.variable("x");
  const pair = E.operation("f", shared, shared).data;
  assert(same([...F.walk(pair)].map((e) => e.form().kind), ["operation", "variable"])); // shared: visited once
  assert(text([...F.walk(basic)].slice(0, 3).map((e) => Object.fromEntries(e.form().attributes))) ===
    text([{ name: "a" }, { name: "get" }, { name: "this" }]));
  assert(F.fold(pair, (_node, results: number[]) => 1 + results.reduce((a, b) => a + b, 0)) === 3); // but folded into each use
  const calls: unknown[] = [];
  F.fold(pair, (term) => calls.push(term));
  assert(calls.length === 2); // once per term
  assert(text(F.fold(new E.OfLet.Data("x", null, E.OfLiteral.resolve(1n)), (_node, results) => results)) === "[null,[]]");
  const loop = new E.OfOperation.Builder().name("f").create();
  new E.OfOperation.Builder(loop).arguments(loop).update();
  assert(same([...F.walk(loop)], [loop]));
  raises(ValueError, () => F.fold(loop, () => null), "the expression contains a cycle");
  const other = new E.OfOperation.Builder().name("f").create();
  new E.OfOperation.Builder(other).arguments(other).update();
  assert(F.same(loop, other) && !F.same(loop, pair));
  for (const [a, b, expected] of [[1n, 1n, true], [1n, 1.0, false], [NaN, NaN, true], [-0, 0, false], [0, 0, true],
    ["x", "x", true], [new Uint8Array([120]), new Uint8Array([120]), true], [new Uint8Array([120]), new Uint8Array([121]), false]] as const) {
    assert(F.same(E.OfLiteral.resolve(a), E.OfLiteral.resolve(b)) === expected, String(a));
  }
  assert(F.same(basic, E.DIALECT.make(basic.form())) && F.same(new E.OfLet.Data("x"), new E.OfLet.Data("x")));
  assert(!F.same(new E.OfLet.Data("x"), E.let_("x", 1n, 2n).data) && !F.same(E.OfLiteral.resolve(1n), P.constant(1n)));
  assert(!F.same(E.operation("f", 1n).data, E.operation("f", 1n, 2n).data));
  assert(!F.same(E.operation("f").data, new E.OfOperation.Data())); // attributes differ
}

## FRM-04 · Every dialect's meta-schemas and builders are derived from its kinds, and round-trip

In [ ]:
{
  for (const dialect of DIALECTS.slice(1)) {
    for (const kind of dialect.kinds().values()) {
      const name = kind.NAME;
      assert(name.startsWith(`Expressions.${dialect.name()}.Of`) && store.schema(name) === kind.Schema
        && dialect.Builders.schema(name) === kind.Schema);
      assert(kind.Schema.validate().length === 0 && dialect.Builders.name_of(kind.Schema) === name);
    }
    assert(same(dialect.Schema.branches.map((b) => b.type), [...dialect.kinds().values()].map((k) => k.Schema)));
    const expression = samples[dialect.name()];
    const schema = dialect.schema_of(expression);
    const codecs: [(schema: any, value: any) => unknown, (builders: any) => { Reachable(schema: any, text: any): any }][] = [
      [JSON.ToJSON(store).Reachable, JSON.FromJSON], [YAML.ToYAML(store).Reachable, YAML.FromYAML]];
    for (const [encode, decode] of codecs) {
      const back = decode(dialect.Builders).Reachable(schema, encode(schema, expression));
      assert(F.same(back, expression) && back.validate({ bound: ["this"] }).length === 0);
    }
    assert(same(dialect.Schema.branches.map((b) => b.name), [...dialect.kinds().keys()])); // named by the kinds' tags
  }
  assert(same([...kindOf(M.DIALECT, "binary").Schema.properties.keys()], ["kind", "operator"]));
  assert(same([...kindOf(X.DIALECT, "constant").Schema.properties.keys()], ["kind", "int", "float", "str", "bool"]));
  assert(same([...kindOf(P.DIALECT, "name").Schema.adjacencies.keys()], ["used_by"]));
  assert(same([...P.DIALECT.kinds().values()].map((k) => k.NAME).slice(-6), [
    "Expressions.Python.OfUnaryOp", "Expressions.Python.OfIfExp", "Expressions.Python.OfGeneratorExp", "Expressions.Python.OfLet",
    "Expressions.Python.OfImport", "Expressions.Python.OfImportFrom"]));
  raises(TypeError, () => E.DIALECT.schema_of(3n), "not an expression: 3");
  raises(TypeError, () => JSON.ToJSON(store).Reachable(kindOf(X.DIALECT, "constant").Schema, new (kindOf(X.DIALECT, "constant"))(bytes(120))),
    "a constant cannot hold a bytes");

  const Builder = X.DIALECT.builders.get("infix") as any;
  const infix = new Builder().set("operator", "+").argument("left", 1n).argument("right", 2n).argument("left", 3n).create();
  assert(infix.left.value === 3n && infix.right.value === 2n
    && new (X.DIALECT.builders.get("constant") as any)().set("value", 1n).create().value === 1n);
  const CallBuilder = P.DIALECT.builders.get("call") as any;
  const call = new CallBuilder().arguments(1n, (a: any) => a.select(2n)).argument("function", P.name("f")).create();
  assert(call.function.name === "f" && same(call.arguments.map((a: any) => a.value), [1n, 2n])); // a slot, then variadic arguments
  const again = new CallBuilder(call).arguments(3n).clone();
  assert(again.function === call.function && same(again.arguments.map((a: any) => a.value), [1n, 2n, 3n]));
  raises(KeyError, () => new Builder().set("name", "x"), "an infix has no attribute 'name'");
  raises(TypeError, () => new Builder().arguments(1n), "an infix has no variadic arguments");
  raises(KeyError, () => new Builder().argument("operand", 1n), "an infix has no argument 'operand'");

  /** Adds an `arguments` entry through the protocol, as FromPlain does. */
  const add = (builder: any, index: bigint | null, target: unknown) => {
    builder.adjacency("arguments", (a: any) => a.add((entry: any) => {
      entry.link("argument", (k: any) => k.set(target));
      if (index !== null) set_property(entry, "index", index);
    }));
    return builder;
  };
  raises(ValueError, () => add(new Builder().set("operator", "+"), 2n, infix).create(),
    "an infix's left is argument 0 and its right argument 1, got index 2");
  const FieldBuilder = M.DIALECT.builders.get("field") as any;
  raises(ValueError, () => add(new FieldBuilder().set("name", "x"), null, M.constant(1n)).create(),
    "a field's value is argument 0, got index None");
  raises(TypeError, () => add(new FieldBuilder().set("name", "x"), 0n, infix).create(),
    "an argument must be an expression, got _Infix"); // an expression of its own dialect
  const Three = { KIND: "if", SLOTS: ["condition", "then", "otherwise"] }; // a kind with three slots, to check the message
  assert(F.slotsMessage(Three, 5n) === "an if's condition is argument 0, its then argument 1 and its otherwise argument 2, got index 5");
}

## FRM-05 · Validation follows each kind's role, in every dialect

In [ ]:
{
  assert(text(DIALECTS.map((d) => samples[d.name()].validate({ bound: ["this"], core: true }))) === "[[],[],[],[],[],[],[]]");
  assert(same(P.subscript(P.name("t"), "x").validate(), ["value: name 't' is not bound"]));
  assert(same(P.call("f", 1n).validate(), ["function: name 'f' is not bound"])); // a slot, then variadic arguments
  assert(same(P.call(P.name("f"), P.name("x")).validate({ bound: ["f"] }), ["argument 0: name 'x' is not bound"]));
  assert(P.import_("numpy", P.name("np"), "np").validate().length === 0); // an import binds a name; its alias is optional
  assert(same(P.import_("numpy", P.name("np")).validate(), ["body: name 'np' is not bound"]));
  assert(same(new (kindOf(P.DIALECT, "import"))("numpy").validate(), ["an import needs a body"]));
  assert(same(P.name("not a name").validate(), ["a name must be an identifier, got 'not a name'"])); // a kind's own check
  assert(X.cell("A1").validate().length === 0 && X.cell("B2", "Sheet1").validate({ core: true }).length === 0); // cells need no binding
  assert(same(new (kindOf(M.DIALECT, "field"))().validate(), ["a field needs a name", "a field needs a value"]));
  assert(same(new (kindOf(M.DIALECT, "binary"))("&&", M.constant(1n)).validate(), ["a binary needs a right"]));
  assert(M.binary("**", 1n, 2n).validate().length === 0 && same(M.binary("**", 1n, 2n).validate({ core: true }), ["'**' is not a core operation"]));
  assert(M.field(M.identifier("s"), "any name").validate({ bound: ["s"], core: true }).length === 0); // a field's vocabulary is open
  assert(same(X.function("NOT", true, false).validate(), ["NOT takes 1 arguments, got 2"]) && X.function("AND", true).validate().length === 0); // AND: any number
  assert(same(X.let_("a", X.name("a"), X.name("a")).validate(), ["value: name 'a' is not bound"]));
  assert(same(new (kindOf(X.DIALECT, "constant"))(bytes(120)).validate(), ["a constant cannot hold a bytes"]));
  assert(same(new (kindOf(X.DIALECT, "constant"))([]).validate(), ["a constant must hold a native value, got list"]));
  assert(same(new (kindOf(M.DIALECT, "call"))(7n).validate(), ["a call's function must be a str, got int"]));
  assert(same(new E.OfOperation.Data("add", [1n]).validate(), ["add takes 2 arguments, got 1", "argument 0: not an expression: 1"]));
}

## FRM-06 · Domains and signatures

In [ ]:
{
  const [Int, Float, Str] = [new D.OfTypes("int", BigInt), new D.OfTypes("float", Number), new D.OfTypes("str", String)];
  const Odd = new D.OfValues("odd", (v) => Number(v) % 2 === 1);
  const Number_ = new D.OfUnion(Int, Float);
  assert(Int.contains(1n) && !Int.contains(true) && !Int.contains(1.0) && String(Int) === "int");
  assert(new D.OfTypes("bool", Boolean).contains(true) && !new D.OfTypes("bool", Boolean).contains(1n));
  assert(Str.contains("a") && !Str.contains(new Uint8Array([97])));
  assert(Number_.name() === "int | float" && String(Number_) === "int | float" && Number_.contains(1.5) && !Number_.contains("1"));
  assert(Number_.includes(Int) && Number_.includes(new D.OfUnion(Float, Int)) && !Int.includes(Number_));
  assert(new D.OfTypes("num", BigInt, Number).includes(Number_) && !Number_.includes(Str) && !Number_.includes(D.Anything));
  assert(Odd.contains(3n) && Odd.includes(Odd) && !Odd.includes(Int) && Odd.includes(new D.OfUnion(Odd)) && String(Odd) === "odd");
  assert(D.Anything.includes(Number_) && D.Anything.contains(null) && String(D.Anything) === "any" && D.Anything.name() === "any");
  assert(D.overlaps(Int, D.Anything) && D.overlaps(Number_, Int) && !D.overlaps(Int, Str));
  const f = new D.Function([Int, Str], Float);
  assert(f.arity() === 2 && f.result([Int, Str]) === Float && f.result([D.Anything, Str]) === Float);
  assert(f.result([Str, Str]) === null && f.result([Int]) === null && f.describe() === "(int, str) -> float");
  assert(f.exact([Int, Str]) && !f.exact([D.Anything, Str]));
  const sameOf = new D.Same(2, [Int, Float]);
  assert(sameOf.result([Int, Int]) === Int && sameOf.result([Int, Float]) === null && sameOf.result([Int]) === null);
  assert(sameOf.result([D.Anything, D.Anything])?.name() === "int | float" && sameOf.result([D.Anything, Float]) === Float);
  assert(sameOf.describe() === "(T, T) -> T for T in int | float" && new D.Same(1, [Int], Str).describe() === "(T) -> str for T in int");
  assert(new D.Same(2, [Int, Float], Str).result([D.Anything, D.Anything]) === Str); // one result, however many match
  assert(new D.Opaque().arity() === -1 && new D.Opaque(Int).result([Str, Str, Str]) === Int && new D.Opaque().describe() === "(...) -> any");
  const either = new D.Either(2);
  assert(either.arity() === 2 && either.result([Int, Int]) === Int && either.result([Int, Str])?.name() === "int | str");
  assert(either.result([Int]) === null && either.describe() === "(T, T) -> T");
  assert(sameOf.exact([Int, Int]) && !sameOf.exact([Int, Float]) && !sameOf.exact([Int]));
  const over = new D.Overloaded(new D.Same(2, [Int]), new D.Function([Number_, Number_], Float));
  assert(over.arity() === 2 && over.result([Int, Int]) === Int && over.result([Int, Float]) === Float);
  assert(over.result([D.Anything, Int])?.name() === "int | float" && over.result([Str, Str]) === null);
  assert(over.describe() === "(T, T) -> T for T in int | (int | float, int | float) -> float");
}

## FRM-07 · `infer` finds an expression's domain from its vocabulary's signatures

In [ ]:
import * as Basic from "@mbse/expressions/Dialects/Basic";
{
  const BD = Basic.Domains;
  assert(E.DIALECT.infer(basic, { this: BD.Object }) === BD.Bool);
  assert(E.DIALECT.infer(E.literal(1n).add(2n)) === BD.Int && E.DIALECT.infer(E.literal(1.5).neg()) === BD.Float);
  assert(E.DIALECT.infer(self.age, { this: BD.Object }) === D.Anything);
  assert(E.DIALECT.infer(self.age.add(1n), { this: BD.Object }) === BD.Int); // only int + int applies
  assert(E.DIALECT.infer(E.operation("contains", 1n, 2n)) === D.Anything); // an extension
  assert(E.DIALECT.infer(E.let_("a", 1n, E.let_("b", "s", E.variable("a")))) === BD.Int);
  raises(TypeError, () => E.DIALECT.infer(E.literal(1n).add(1.5)), "add cannot take (int, float); it takes (T, T) -> T for T in int | float");
  raises(TypeError, () => E.DIALECT.infer(E.literal(1n).has("x")), "has cannot take (int, str); it takes (object | record, str) -> bool");
  raises(ValueError, () => E.DIALECT.infer(self), "cannot infer the domain of an invalid expression: variable 'this' is not bound");
  assert(BD.of(E.OfLiteral.resolve(1n)) === BD.Object && BD.of(bytes()) === BD.Bytes);
  raises(TypeError, () => BD.of([]), "a list is not a value of the Basic dialect");
  assert(M.DIALECT.infer(samples.Matlab, { this: D.Anything }).name() === "logical");
  assert(X.DIALECT.infer(samples.Excel, { this: D.Anything }).name() === "logical");
  assert(L.DIALECT.infer(samples.Latex, { this: D.Anything }).name() === "truth");
  assert(P.DIALECT.infer(samples.Python, { this: D.Anything }) === D.Anything); // a call gives what is not known
  assert(P.DIALECT.infer(P.boolop("and", 1n, "a")).name() === "int | str"); // Python's and gives an operand
  assert(P.DIALECT.infer(P.import_("math", P.binop("+", 1n, true))).name() === "int");
}

## FRM-08 · The interpreter evaluates by role and raises what validation reports

In [ ]:
{
  const interpreter = new FV.Interpreter(M.DIALECT, new Map<string, any>([["binary", new Map()], ["unary", new Map()],
    ["call", new Map()], ["field", () => null]]));
  raises(TypeError, () => interpreter.run(3n), "not an expression: 3");
  raises(ValueError, () => interpreter.run(M.constant(null as any)), "a constant needs a value");
  raises(KeyError, () => interpreter.run(M.identifier("x")), "identifier 'x' is not bound");
  raises(NotImplementedError, () => interpreter.run(M.binary("&&", 1n, 2n)), "'&&' is not a core operation");
  assert(interpreter.run(M.field(M.identifier("s"), "f"), { s: 1n }) === null);
  raises(TypeError, () => new FV.Interpreter(M.DIALECT, new Map([["call", new Map([["isfield", null as any]])]])).run(M.call("isfield", 1n)),
    "isfield takes 2 arguments, got 1");
  const cycle = new (kindOf(X.DIALECT, "let"))("a", X.constant(1n));
  cycle.body = cycle;
  raises(ValueError, () => XV.OfAny(cycle), "the expression contains a cycle");
  raises(ValueError, () => XV.OfAny(new (kindOf(X.DIALECT, "let"))("a", X.constant(1n))), "a let needs a body");
  raises(ValueError, () => MV.OfAny(new (kindOf(M.DIALECT, "import"))("pkg.*")), "an import needs a body");

  const scope = new SY.Variables({ x: 1n });
  assert(conformance_problems(SY.Variables, "Scope").length === 0 && scope.lookup(E.variable("x").data) === 1n);
  const inner = scope.bind("x", 2n);
  assert(inner.lookup(E.variable("x").data) === 2n && scope.lookup(E.variable("x").data) === 1n); // bind makes a new scope
  raises(KeyError, () => scope.lookup(E.variable("y").data), "variable 'y' is not bound");
  raises(NotImplementedError, () => scope.enter(M.import_("pkg.*", 1n)), "Variables has no imports");
  assert(V.OfAny(E.variable("x"), inner) === 2n); // an evaluator takes a scope as well as an object of variables
  raises(NotImplementedError, () => interpreter.run(M.call("dist", 1n, 2n)), "'dist' is not a core operation"); // no extension: the vocabulary only
}

## FRM-09 · Symbolics: the names an expression needs, and the imports it declares

In [ ]:
{
  const sorted = (names: Set<string>) => [...names].sort();
  assert(same(sorted(SY.free(basic)), ["this"]) && SY.free(E.literal(1n)).size === 0);
  assert(same(sorted(SY.free(E.let_("a", E.variable("a"), E.variable("a")))), ["a"])); // the name is bound in the body only
  assert(same(sorted(SY.free(samples.Python)), ["this"])); // np, f and a are bound by imports and the let
  assert(same(sorted(SY.free(P.call("hasattr", P.name("x"), "k"))), ["x"])); // builtins are ambient
  assert(same(sorted(SY.free(X.infix("+", X.cell("A1"), X.name("r")))), ["r"])); // cells are resolved by the workbook, not bound
  assert(same(sorted(SY.free(M.import_("pkg.*", M.identifier("x")))), ["x"])); // MATLAB's imports bind functions, not variables
  const knot = new E.OfOperation.Builder().name("f").create();
  new E.OfOperation.Builder(knot).arguments(knot, E.variable("x")).update();
  assert(same(sorted(SY.free(knot)), ["x"])); // cycles end the search
  assert(same(sorted(SY.free(new E.OfLet.Data("a", null, E.variable("b").data))), ["b"])); // empty slots are skipped
  assert(same(SY.imports(samples.Python).map((i) => i.kind().KIND), ["import", "importfrom"]) && SY.imports(basic).length === 0);
  assert(same(SY.imports(samples.Matlab).map((i) => i.name), ["pkg.*"]));
  for (const [error, name] of [[FE.NameError, "NameError"], [FE.ImportError, "ImportError"], [FE.OverflowError, "OverflowError"],
    [FE.ZeroDivisionError, "ZeroDivisionError"]] as const) {
    assert(new error("m") instanceof Error && new error("m").name === name); // Python's exceptions, by the same names
  }
}

## FRM-10 · A dialect may extend another: its kinds are the base's and its own

In [ ]:
{
  /** `twice(x)`, x + x: the one kind of an extension of Basic. */
  class Twice_ extends F.Term {
    static override KIND = "twice";
    static override ROLE = F.APPLICATION;
    static override SLOTS = ["operand"];
    declare operand: any;

    constructor(operand: unknown = null) {
      super(operand);
    }
  }

  const BD = (await import("@mbse/expressions")).Domains;
  const Twice = new F.Declared("Twice", [Twice_ as unknown as F.TermClass], { domain_of: BD.of, extends: E.DIALECT });
  const x = E.variable("x");
  const mixed = x.add(new Twice_(x.data)).data; // a writer of the base holds the extension's term
  assert(E.writer(new Twice_(x.data)).add(1n).data.arguments[0] instanceof Twice_); // and writes it
  assert(mixed.arguments[1] instanceof Twice_ && E.DIALECT.terms().includes(Twice_ as never) && E.DIALECT.extensions.includes(Twice));
  assert(Twice.base === E.DIALECT && same(Twice.classes, [...E.DIALECT.classes, Twice_]) && E.DIALECT.base === null);
  assert(same([...Twice.kinds().keys()], [...E.DIALECT.kinds().keys(), "twice"]) && same(Twice.terms(), Twice.classes));
  assert(Twice.validate(mixed, { bound: ["x"] }).length === 0);
  const problems = E.DIALECT.validate(mixed, { bound: ["x"] }); // the base cannot validate what holds the extension's terms
  assert(problems.length === 1 && problems[0]!.startsWith("argument 1: not an expression: "));
  const interpreter = new FV.Interpreter(Twice, new Map<string, unknown>([["operation", V.OPERATIONS], ["quantifier", V.QUANTIFIERS],
    ["twice", (thunks: (() => unknown)[]) => (thunks[0]!() as bigint) * 2n]]) as never);
  assert(interpreter.run(mixed, { x: 3n }) === 9n);
  assert(Twice.schemas.has("Expressions.Twice.OfTwice") && Twice.schemas.has("Expressions.OfOperation"));
  assert(same(Twice.Schema.branches.map((b) => b.name), [...E.DIALECT.kinds().keys(), "twice"]));
  const text = JSON.ToJSON(Twice.Builders).Reachable(Twice.schema_of(mixed), mixed);
  assert(F.same(JSON.FromJSON(Twice.Builders).Reachable(Twice.schema_of(mixed), text), mixed)); // the base reads it too

  /** `recur(d)`: refers to a definition `d`, which may refer back to it. */
  class Recur_ extends F.Term {
    static override KIND = "recur";
    static override ROLE = F.APPLICATION;
    static override SLOTS = ["definition"];
    static override REFERS = new Set(["definition"]);
    declare definition: any;

    constructor(definition: unknown = null) {
      super(definition);
    }
  }

  const Recur = new F.Declared("Recur", [Recur_ as unknown as F.TermClass], { domain_of: BD.of, extends: E.DIALECT });
  const definition = new E.OfOperation.Builder().name("f").create();
  new E.OfOperation.Builder(definition).arguments(new Recur_(definition)).update();
  assert(Recur.validate(new Recur_(definition)).length === 0); // recursion, through a slot that refers: not a cycle
  const looped = new Twice_();
  looped.operand = looped;
  assert(same(Twice.validate(looped), ["operand: the expression contains a cycle"])); // through any other slot, a cycle

  class Literal_ extends F.Term {
    static override KIND = "literal";
    static override ROLE = F.APPLICATION;
  }

  raises(ValueError, () => new F.Declared("Clash", [Literal_ as unknown as F.TermClass], { domain_of: BD.of, extends: E.DIALECT }),
    "Clash would declare 'literal', which Basic has");
}